# 03. 얼굴 분석 — 모델 여러 개를 잇기

얼굴 찾기(01) → 얼굴만 잘라서 → 나이·성별 / 감정 / 머리 방향 모델에 각각 넣는다.
모델마다 **입력 크기**와 **출력 이름**이 다르다 — 모델 설명을 읽고 맞추는 연습.

| 모델 | 입력 | 출력 |
|---|---|---|
| face-detection-retail-0005 | 300 x 300 BGR | `[1,1,N,7]` |
| age-gender-recognition-retail-0013 | 62 x 62 BGR | `age_conv3` (나이 ÷ 100), `prob` ([여, 남]) |
| emotions-recognition-retail-0003 | 64 x 64 BGR | 5종 확률: 무표정 · 행복 · 슬픔 · 놀람 · 화남 |
| head-pose-estimation-adas-0001 | 60 x 60 BGR | `angle_y_fc` · `angle_p_fc` · `angle_r_fc` (도) |

## 1. 준비 — 모델 네 개 올리기

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

def pick(*prefer):
    """먼저 적은 장치부터, 이 PC 에 있는 것을 고른다."""
    for d in prefer:
        if d in core.available_devices:
            return d
    return "CPU"

DEVICE = pick("NPU", "GPU", "CPU")                  # 작은 모델들 — NPU 가 잘 맞는다
F = MODELS / "face"
det = core.compile_model(F / "face-detection-retail-0005.xml", DEVICE)
agegen = core.compile_model(F / "age-gender-recognition-retail-0013.xml", DEVICE)
emo = core.compile_model(F / "emotions-recognition-retail-0003.xml", DEVICE)
pose = core.compile_model(F / "head-pose-estimation-adas-0001.xml", DEVICE)
for name, c in [("얼굴", det), ("나이·성별", agegen), ("감정", emo), ("머리 방향", pose)]:
    print(f"{name:6s} 입력 {c.input(0).shape}  출력 {[o.any_name for o in c.outputs]}")

## 2. 사진

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 3. 얼굴 찾기 → 잘라 내기

In [ ]:
def blob(bgr, size):
    return cv2.resize(bgr, (size, size)).transpose(2, 0, 1)[None].astype(np.float32)

def find_faces(bgr, conf=0.5, margin=0.2):
    h, w = bgr.shape[:2]
    dets = det(blob(bgr, 300))[det.output(0)].reshape(-1, 7)
    out = []
    for image_id, _, score, x1, y1, x2, y2 in dets:
        if image_id < 0:
            break
        if score < conf:
            continue
        x1, y1, x2, y2 = x1 * w, y1 * h, x2 * w, y2 * h
        mx, my = (x2 - x1) * margin, (y2 - y1) * margin        # 머리카락·턱까지 조금 넉넉히
        box = (int(max(0, x1 - mx)), int(max(0, y1 - my)), int(min(w, x2 + mx)), int(min(h, y2 + my)))
        out.append((box, float(score)))
    return out

found = find_faces(frame)
print("얼굴", len(found), "개")
for (x1, y1, x2, y2), s in found:
    show(frame[y1:y2, x1:x2], 160)

## 4. 잘라 낸 얼굴마다 세 모델

In [ ]:
EMO = ["무표정", "행복", "슬픔", "놀람", "화남"]

def analyze(face):
    r = agegen(blob(face, 62))
    age = float(r[agegen.output("age_conv3")].squeeze()) * 100
    female, male = r[agegen.output("prob")].squeeze()

    p = emo(blob(face, 64))[emo.output(0)].squeeze()

    r = pose(blob(face, 60))
    yaw = float(r[pose.output("angle_y_fc")].squeeze())      # 좌우 (도)
    pitch = float(r[pose.output("angle_p_fc")].squeeze())    # 위아래
    roll = float(r[pose.output("angle_r_fc")].squeeze())     # 갸웃
    return {"나이": round(age), "성별": "여" if female > male else "남",
            "감정": EMO[int(p.argmax())], "감정 확률": np.round(p, 2),
            "좌우": round(yaw), "위아래": round(pitch), "갸웃": round(roll)}

view = frame.copy()
for i, ((x1, y1, x2, y2), s) in enumerate(found):
    info = analyze(frame[y1:y2, x1:x2])
    print(f"얼굴 {i}:", info)
    cv2.rectangle(view, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.putText(view, f"#{i} age {info['나이']} yaw {info['좌우']}", (x1, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)
show(view)

글자를 사진에 쓸 때 `cv2.putText` 는 한글을 못 쓴다 — 그래서 사진에는 영어·숫자만 쓰고 한글은 표로 찍었다.
(한글을 사진에 쓰려면 PIL 의 `ImageDraw.text` 와 한글 글꼴을 쓴다.)

## 5. 얼굴 하나에 몇 ms?

In [ ]:
if found:
    (x1, y1, x2, y2), _ = found[0]
    face = frame[y1:y2, x1:x2]
    for name, fn in [("찾기", lambda: find_faces(frame)), ("분석 셋", lambda: analyze(face))]:
        t0 = time.perf_counter()
        for _ in range(20):
            fn()
        print(f"{name:6s} {(time.perf_counter() - t0) / 20 * 1000:6.2f} ms  ({DEVICE})")

## 해 볼 것
1. `margin` 을 0 과 0.5 로 바꾸면 나이 · 감정이 달라지나? 모델이 배운 얼굴 자르기와 맞아야 한다.
2. `DEVICE` 를 바꿔 5번 시간을 비교해 보자.
3. 웹캠을 여러 번 찍어(`frame = grab()`) 표정을 바꿔 가며 감정 확률이 어떻게 움직이는지 보자.
4. 머리 방향으로 "정면 / 왼쪽 / 오른쪽 / 위 / 아래" 를 판정하는 함수를 만들어 보자 (예: `|좌우| < 15` 이면 정면).